In [ ]:
# NotY3dGenAI - Advanced 3D Model Generator for Google Colab
# Using only reliable libraries that work in Colab

import os
import sys
import time
import torch
import numpy as np
from PIL import Image
import plotly.graph_objects as go
from IPython.display import display, HTML, clear_output, FileLink
import ipywidgets as widgets
from google.colab import output, drive, files
import json
import warnings
warnings.filterwarnings('ignore')

# Mount Google Drive
drive.mount('/content/drive')

# Install minimal required packages


# Import after installation
import cv2
from scipy import ndimage
from skimage import measure, filters, morphology
from stl import mesh as stl_mesh
from pathlib import Path

print("✅ All packages installed successfully!")

# Create directories
Path("/content/noty3d_models").mkdir(exist_ok=True)
Path("/content/drive/MyDrive/NotY3D_Models").mkdir(exist_ok=True)

class NotY3dGenAI:
    def __init__(self):
        self.device = self.setup_device()
        self.setup_models()
        self.create_ui()
        
    def setup_device(self):
        """Setup best available device"""
        if torch.cuda.is_available():
            device = torch.device("cuda")
            print(f"✅ GPU detected: {torch.cuda.get_device_name(0)}")
            return device
        else:
            print("✅ Using CPU (will still work well)")
            return torch.device("cpu")
    
    def setup_models(self):
        """Initialize models"""
        print("🔄 Loading Stable Diffusion...")
        
        try:
            from diffusers import StableDiffusionPipeline
            
            # Use a smaller, faster model for Colab
            model_id = "runwayml/stable-diffusion-v1-5"
            
            self.pipeline = StableDiffusionPipeline.from_pretrained(
                model_id,
                torch_dtype=torch.float32,
                low_cpu_mem_usage=True,
                safety_checker=None
            )
            self.pipeline.to(self.device)
            print("✅ Models loaded successfully!")
        except Exception as e:
            print(f"⚠️ Could not load full model: {e}")
            print("Using fallback image generation...")
            self.pipeline = None
    
    def generate_single_image(self, prompt, seed=42):
        """Generate image from prompt"""
        if self.pipeline is None:
            # Fallback: generate gradient image
            img = Image.new('RGB', (512, 512))
            from PIL import ImageDraw
            draw = ImageDraw.Draw(img)
            
            # Create artistic gradient
            for i in range(512):
                color = int(100 + 155 * i / 512)
                draw.line([(0, i), (512, i)], fill=(color, color, color))
            
            # Add text
            draw.text((50, 256), prompt[:50], fill=(255, 255, 255))
            return img
        
        torch.manual_seed(seed)
        with torch.autocast("cuda" if torch.cuda.is_available() else "cpu"):
            image = self.pipeline(
                prompt,
                num_inference_steps=25,
                guidance_scale=7.0,
                height=512,
                width=512
            ).images[0]
        
        return image
    
    def create_height_map(self, image):
        """Create height/depth map from image"""
        # Convert to grayscale
        img_array = np.array(image.convert('L'))
        
        # Apply edge detection for structure
        edges = cv2.Canny(img_array, 50, 150)
        
        # Create depth from intensity
        depth = img_array.astype(np.float32) / 255.0
        
        # Add edge emphasis
        depth = depth + edges.astype(np.float32) * 0.3
        
        # Apply Gaussian blur for smoothness
        depth = cv2.GaussianBlur(depth, (15, 15), 0)
        
        # Normalize
        depth = (depth - depth.min()) / (depth.max() - depth.min())
        
        return depth
    
    def create_mesh_from_depth(self, depth_map, image, resolution=100, smooth=True):
        """Create 3D mesh from depth map using marching cubes"""
        
        # Resample depth map to target resolution
        h, w = depth_map.shape
        scale_h = resolution / h
        scale_w = resolution / w
        
        depth_resized = cv2.resize(depth_map, (resolution, resolution))
        
        # Create 3D volume
        volume = np.zeros((resolution, resolution, resolution))
        
        # Fill volume based on depth
        for i in range(resolution):
            for j in range(resolution):
                depth_value = depth_resized[i, j]
                z_height = int(depth_value * (resolution - 1))
                volume[i, j, :z_height] = 1.0
        
        # Apply smoothing if requested
        if smooth:
            volume = ndimage.gaussian_filter(volume, sigma=1.5)
        
        # Apply marching cubes to extract mesh
        try:
            verts, faces, normals, values = measure.marching_cubes(volume, level=0.5)
        except:
            # Alternative method
            verts, faces, normals, values = measure.marching_cubes(volume, level=0.3)
        
        # Normalize vertices
        verts = verts / resolution
        verts = verts - 0.5  # Center at origin
        
        # Get colors from original image
        img_array = np.array(image.resize((resolution, resolution)))
        
        # Assign colors to vertices based on position
        vertex_colors = []
        for v in verts:
            x = int(v[0] * resolution) % resolution
            y = int(v[1] * resolution) % resolution
            color = img_array[y, x, :3] / 255.0
            vertex_colors.append(color)
        
        return verts, faces, np.array(vertex_colors)
    
    def save_as_obj(self, verts, faces, colors, filename):
        """Save mesh as OBJ file with colors"""
        with open(filename, 'w') as f:
            f.write("# NotY3dGenAI Generated Model\n")
            f.write(f"o generated_model\n")
            
            # Write vertices with colors
            for i, (v, c) in enumerate(zip(verts, colors)):
                f.write(f"v {v[0]} {v[1]} {v[2]} {c[0]} {c[1]} {c[2]}\n")
            
            # Write faces
            for face in faces:
                f.write(f"f {face[0]+1} {face[1]+1} {face[2]+1}\n")
    
    def save_as_ply(self, verts, faces, colors, filename):
        """Save mesh as PLY file"""
        with open(filename, 'w') as f:
            f.write("ply\n")
            f.write("format ascii 1.0\n")
            f.write(f"element vertex {len(verts)}\n")
            f.write("property float x\n")
            f.write("property float y\n")
            f.write("property float z\n")
            f.write("property uchar red\n")
            f.write("property uchar green\n")
            f.write("property uchar blue\n")
            f.write(f"element face {len(faces)}\n")
            f.write("property list uchar int vertex_indices\n")
            f.write("end_header\n")
            
            # Write vertices
            for v, c in zip(verts, colors):
                f.write(f"{v[0]} {v[1]} {v[2]} {int(c[0]*255)} {int(c[1]*255)} {int(c[2]*255)}\n")
            
            # Write faces
            for face in faces:
                f.write(f"3 {face[0]} {face[1]} {face[2]}\n")
    
    def simplify_mesh(self, verts, faces, target_faces=10000):
        """Simplify mesh by reducing face count"""
        if len(faces) <= target_faces:
            return verts, faces
        
        # Simple simplification: sample every nth face
        step = len(faces) // target_faces
        simplified_faces = faces[::step]
        
        # Get unique vertices used in simplified faces
        used_vertices = set()
        for face in simplified_faces:
            used_vertices.update(face)
        
        # Remap vertex indices
        vertex_map = {old: new for new, old in enumerate(sorted(used_vertices))}
        new_verts = verts[list(used_vertices)]
        new_faces = np.array([[vertex_map[v] for v in face] for face in simplified_faces])
        
        return new_verts, new_faces
    
    def save_model(self, verts, faces, colors, prompt, format="obj"):
        """Save model in requested format"""
        timestamp = int(time.time())
        safe_prompt = "".join(c for c in prompt[:30] if c.isalnum() or c in (' ', '-', '_')).rstrip()
        filename = f"noty3d_{safe_prompt}_{timestamp}.{format}"
        
        # Save locally
        local_path = f"/content/noty3d_models/{filename}"
        
        if format == "obj":
            self.save_as_obj(verts, faces, colors, local_path)
        elif format == "ply":
            self.save_as_ply(verts, faces, colors, local_path)
        else:
            self.save_as_obj(verts, faces, colors, local_path)
        
        # Save to Drive
        drive_path = f"/content/drive/MyDrive/NotY3D_Models/{filename}"
        if format == "obj":
            self.save_as_obj(verts, faces, colors, drive_path)
        elif format == "ply":
            self.save_as_ply(verts, faces, colors, drive_path)
        else:
            self.save_as_obj(verts, faces, colors, drive_path)
        
        # Save metadata
        metadata = {
            "prompt": prompt,
            "timestamp": timestamp,
            "format": format,
            "vertices": len(verts),
            "faces": len(faces),
            "device": str(self.device)
        }
        
        with open(f"/content/noty3d_models/{filename}.json", "w") as f:
            json.dump(metadata, f, indent=2)
        
        return local_path, drive_path
    
    def create_ui(self):
        """Create interactive UI"""
        
        # Style
        display(HTML("""
        <style>
            .noty3d-title {
                font-size: 42px;
                font-weight: bold;
                background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                -webkit-background-clip: text;
                -webkit-text-fill-color: transparent;
                text-align: center;
                margin-bottom: 20px;
            }
            .info-box {
                background: linear-gradient(135deg, #667eea 20%, #764ba2 80%);
                color: white;
                padding: 15px;
                border-radius: 10px;
                margin: 10px 0;
                font-size: 14px;
            }
            .control-panel {
                background: #f8f9fa;
                padding: 20px;
                border-radius: 10px;
                margin: 10px 0;
            }
        </style>
        """))
        
        display(HTML('<div class="noty3d-title">🎨 NotY3dGenAI - 3D Model Generator</div>'))
        
        # Info box
        display(HTML(f"""
        <div class="info-box">
            ✅ <b>Device:</b> {self.device}<br>
            🚀 <b>Status:</b> Ready to generate 3D models<br>
            💾 <b>Storage:</b> Models auto-save to Google Drive/NotY3D_Models/
        </div>
        """))
        
        # Prompt input
        self.prompt_input = widgets.Textarea(
            value="a majestic dragon with intricate scales and large wings",
            placeholder="Describe the 3D model you want to generate...",
            description="🎯 Prompt:",
            layout=widgets.Layout(width='100%', height='100px')
        )
        
        # Quality controls
        self.resolution = widgets.IntSlider(
            value=100,
            min=50,
            max=200,
            step=10,
            description="🔍 Mesh Resolution:",
            style={'description_width': 'initial'}
        )
        
        self.quality = widgets.Dropdown(
            options=['High', 'Medium', 'Low'],
            value='Medium',
            description="✨ Quality:",
            style={'description_width': 'initial'}
        )
        
        self.smooth = widgets.Checkbox(
            value=True,
            description="Smooth Mesh",
            style={'description_width': 'initial'}
        )
        
        self.poly_count = widgets.IntSlider(
            value=15000,
            min=5000,
            max=30000,
            step=1000,
            description="🔺 Max Polygons:",
            style={'description_width': 'initial'}
        )
        
        self.output_format = widgets.Dropdown(
            options=['obj', 'ply'],
            value='obj',
            description="📁 Output Format:",
            style={'description_width': 'initial'}
        )
        
        # Generate button
        self.generate_btn = widgets.Button(
            description="🚀 Generate 3D Model",
            button_style='primary',
            layout=widgets.Layout(width='100%', height='50px')
        )
        
        self.status_output = widgets.Output()
        
        # Layout
        controls = widgets.VBox([
            self.prompt_input,
            widgets.HBox([self.resolution, self.quality]),
            widgets.HBox([self.smooth, self.poly_count]),
            self.output_format,
            self.generate_btn,
            self.status_output
        ], layout=widgets.Layout(padding='10px'))
        
        self.viewer_output = widgets.Output()
        
        # Main container
        container = widgets.HBox([
            widgets.VBox([controls], layout=widgets.Layout(width='40%')),
            widgets.VBox([
                widgets.HTML("<h3>📊 3D Viewer</h3>"),
                self.viewer_output
            ], layout=widgets.Layout(width='60%'))
        ])
        
        display(container)
        
        # Bind function
        self.generate_btn.on_click(self.generate_model)
    
    def generate_model(self, btn):
        """Generate 3D model"""
        with self.status_output:
            clear_output()
            start_time = time.time()
            
            print("🎨 Starting 3D model generation...")
            print(f"📝 Prompt: {self.prompt_input.value}")
            print(f"⚙️ Settings: {self.quality.value}, Resolution: {self.resolution.value}")
            
            try:
                # Generate image
                print("📸 Generating image from prompt...")
                image = self.generate_single_image(self.prompt_input.value)
                
                # Display generated image
                img_display = widgets.Image(value=self.image_to_bytes(image), format='png')
                display(img_display)
                
                # Create depth map
                print("🗺️ Creating depth map...")
                depth_map = self.create_height_map(image)
                
                # Create mesh
                print("🔺 Generating 3D mesh...")
                resolution = self.resolution.value
                if self.quality.value == 'Low':
                    resolution = resolution // 2
                elif self.quality.value == 'High':
                    resolution = resolution * 2
                
                verts, faces, colors = self.create_mesh_from_depth(
                    depth_map, image, 
                    resolution=min(resolution, 150),
                    smooth=self.smooth.value
                )
                
                # Simplify if needed
                if len(faces) > self.poly_count.value:
                    print(f"📐 Simplifying mesh ({len(faces)} -> {self.poly_count.value} faces)...")
                    verts, faces = self.simplify_mesh(verts, faces, self.poly_count.value)
                
                # Save model
                print("💾 Saving model...")
                local_path, drive_path = self.save_model(
                    verts, faces, colors,
                    self.prompt_input.value,
                    self.output_format.value
                )
                
                # Time taken
                elapsed = time.time() - start_time
                
                print(f"\n✅ Model generated successfully!")
                print(f"⏱️ Time: {elapsed:.2f} seconds")
                print(f"🔺 Vertices: {len(verts):,}")
                print(f"🔻 Faces: {len(faces):,}")
                print(f"💾 Saved: {local_path}")
                print(f"☁️ Drive: {drive_path}")
                
                # Display 3D model
                with self.viewer_output:
                    clear_output()
                    self.display_3d_model(verts, faces, colors)
                
                # Create download link
                display(HTML(f"<br><a href='{local_path}' download>📥 Click here to download model</a>"))
                
                # Also save to Drive link
                display(HTML(f"<p>✅ Also saved to: <code>{drive_path}</code></p>"))
                
            except Exception as e:
                print(f"❌ Error: {str(e)}")
                import traceback
                traceback.print_exc()
    
    def image_to_bytes(self, image):
        """Convert PIL image to bytes for display"""
        import io
        img_byte_arr = io.BytesIO()
        image.save(img_byte_arr, format='PNG')
        return img_byte_arr.getvalue()
    
    def display_3d_model(self, verts, faces, colors):
        """Display 3D model using plotly"""
        try:
            # Sample vertices for better performance if too many
            if len(verts) > 5000:
                step = len(verts) // 5000
                sampled_verts = verts[::step]
                sampled_colors = colors[::step]
                
                # Recreate faces with sampled vertices
                valid_faces = []
                for face in faces:
                    if all(v % step == 0 for v in face):
                        valid_faces.append([v//step for v in face])
                faces = np.array(valid_faces[:5000])
            else:
                sampled_verts = verts
                sampled_colors = colors
            
            # Create mesh plot
            fig = go.Figure(data=[
                go.Mesh3d(
                    x=sampled_verts[:, 0],
                    y=sampled_verts[:, 1],
                    z=sampled_verts[:, 2],
                    i=faces[:, 0],
                    j=faces[:, 1],
                    k=faces[:, 2],
                    vertexcolor=sampled_colors,
                    intensity=sampled_verts[:, 2],
                    colorscale='Viridis',
                    opacity=0.9,
                    lighting=dict(ambient=0.6, diffuse=0.8, specular=0.3)
                )
            ])
            
            fig.update_layout(
                scene=dict(
                    xaxis_title='X',
                    yaxis_title='Y',
                    zaxis_title='Z',
                    camera=dict(eye=dict(x=1.5, y=1.5, z=1.5)),
                    aspectmode='data'
                ),
                width=700,
                height=500,
                margin=dict(l=0, r=0, b=0, t=0)
            )
            
            fig.show()
            
        except Exception as e:
            print(f"⚠️ Could not display 3D model: {e}")
            print("Model saved successfully though!")

# Initialize app
print("""
╔══════════════════════════════════════════════════════════╗
║                                                          ║
║     🎨 NotY3dGenAI - Professional 3D Generator          ║
║                                                          ║
║     Features:                                            ║
║     • Text to 3D Model Generation                       ║
║     • Full Quality Controls                             ║
║     • Polygon & Resolution Settings                     ║
║     • Multiple Output Formats (OBJ/PLY)                 ║
║     • Auto-save to Google Drive                         ║
║                                                          ║
╚══════════════════════════════════════════════════════════╝
""")

# Create and launch app
app = NotY3dGenAI()

print("\n" + "="*60)
print("✨ NotY3dGenAI is ready!")
print("="*60)
print("\n💡 Tips for best results:")
print("   • Start with 'Medium' quality for testing")
print("   • Use descriptive prompts (e.g., 'detailed dragon with wings')")
print("   • Higher resolution = more detailed but slower")
print("   • Enable 'Smooth Mesh' for organic shapes")
print("\n🎯 Enter your prompt and click 'Generate 3D Model' to begin!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ All packages installed successfully!

╔══════════════════════════════════════════════════════════╗
║                                                          ║
║     🎨 NotY3dGenAI - Professional 3D Generator          ║
║                                                          ║
║     Features:                                            ║
║     • Text to 3D Model Generation                       ║
║     • Full Quality Controls                             ║
║     • Polygon & Resolution Settings                     ║
║     • Multiple Output Formats (OBJ/PLY)                 ║
║     • Auto-save to Google Drive                         ║
║                                                          ║
╚══════════════════════════════════════════════════════════╝

✅ GPU detected: Tesla T4
🔄 Loading Stable Diffusion...
⚠️ Could not load full model: cannot import name '


✨ NotY3dGenAI is ready!

💡 Tips for best results:
   • Start with 'Medium' quality for testing
   • Use descriptive prompts (e.g., 'detailed dragon with wings')
   • Higher resolution = more detailed but slower
   • Enable 'Smooth Mesh' for organic shapes

🎯 Enter your prompt and click 'Generate 3D Model' to begin!


In [1]:
!nvidia-smi

Sun Apr  5 15:50:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----